In [1]:
import os
import sys

os.chdir("..")
sys.path.append("src")

In [2]:
import torch
import pandas as pd
import matplotlib.pyplot as plt

from wildfire_gnn.utils.config import load_yaml_config
from wildfire_gnn.pipelines.gnn_pipeline import GNNPipeline
from wildfire_gnn.features.feature_engineering import add_neighborhood_summary_features, add_degree_feature

In [3]:
with open("configs/gnn_config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg

{'project': {'name': 'wildfire-uncertainty-gnn', 'random_seed': 42},
 'paths': {'graph_data_path': 'data/processed/graph_data.pt',
  'split_path': 'data/processed/baseline_splits_spatial.npz'},
 'outputs': {'checkpoints_dir': 'reports/checkpoints',
  'tables_dir': 'reports/tables',
  'figures_dir': 'reports/figures',
  'logs_dir': 'reports/logs',
  'checkpoint_name': 'uncertainty_gat_best.pt',
  'metrics_filename': 'gnn_metrics.csv',
  'tail_metrics_filename': 'gnn_tail_metrics.csv',
  'predictions_filename': 'gnn_test_predictions.csv',
  'val_predictions_filename': 'gnn_val_predictions.csv',
  'uncertainty_predictions_filename': 'gnn_uncertainty_predictions.csv',
  'calibration_metrics_filename': 'gnn_calibration_metrics.csv',
  'intervention_metrics_filename': 'intervention_metrics.csv',
  'intervention_predictions_filename': 'gnn_intervention_predictions.csv'},
 'model': {'name': 'residual_gat_regressor',
  'input_dim': None,
  'hidden_dim': 96,
  'output_dim': 1,
  'num_layers': 3,

In [6]:
!python scripts/train_gnn.py

Epoch 001 | Train Loss: 0.438678 | Val Loss: 0.400428
Epoch 002 | Train Loss: 0.452405 | Val Loss: 0.379431
Epoch 003 | Train Loss: 0.427410 | Val Loss: 0.351864
Epoch 004 | Train Loss: 0.399931 | Val Loss: 0.352899
Epoch 005 | Train Loss: 0.399671 | Val Loss: 0.356299
Epoch 006 | Train Loss: 0.394677 | Val Loss: 0.349500
Epoch 007 | Train Loss: 0.378517 | Val Loss: 0.345721
Epoch 008 | Train Loss: 0.367603 | Val Loss: 0.346994
Saved checkpoint to: reports/checkpoints\uncertainty_gat_best.pt
Best epoch: 6
Best val loss: 0.345721


In [9]:
!python scripts/evaluate_gnn.py

Saved metrics to: reports/tables\gnn_metrics.csv
Saved tail metrics to: reports/tables\gnn_tail_metrics.csv


In [10]:
metrics_df = pd.read_csv("reports/tables/gnn_metrics.csv")
metrics_df

,rmse,mae,r2,pearson,spearman,tail_rmse,split
0,0.044731,0.030608,-0.872433,-0.061742,-0.107704,0.131182,val
1,0.019310,0.011622,-0.555596,-0.072224,-0.167777,0.123458,test


In [11]:
tail_df = pd.read_csv("reports/tables/gnn_tail_metrics.csv")
tail_df

,split,high_risk_threshold,tail_rmse
0,val,0.1,0.131182
1,test,0.1,0.123458


In [12]:
pred_df = pd.read_csv("reports/tables/gnn_test_predictions.csv")
pred_df.head()

,node_id,split,y_true,y_pred,epistemic_std,aleatoric_std,total_std,row_norm,col_norm
0,1,test,0.023607,0.0,0.198187,1.228981,1.244859,197.0,549.0
1,5,test,0.030925,0.0,0.080399,1.191036,1.193747,269.0,659.0
2,16,test,0.062656,0.0,0.155683,1.210662,1.220631,347.0,495.0
3,18,test,0.003133,0.0,0.114781,1.220087,1.225475,277.0,573.0
4,21,test,0.002632,0.0,0.186202,1.212746,1.226957,177.0,576.0


In [13]:
pred_df.describe()

,node_id,y_true,y_pred,epistemic_std,aleatoric_std,total_std,row_norm,col_norm
count,60115.000000,60115.000000,60115.000000,60115.000000,60115.000000,60115.000000,60115.000000,60115.000000
mean,149880.494835,0.011571,0.000691,0.206116,1.220530,1.241003,512.521317,559.700358
std,86556.969322,0.015482,0.002886,0.092882,0.055622,0.061709,370.095713,239.136936
min,1.000000,0.000031,0.000000,0.006552,0.898134,0.922074,82.000000,239.000000
25%,75106.500000,0.001771,0.000000,0.138994,1.182168,1.197486,231.000000,399.000000
50%,149759.000000,0.005968,0.000000,0.191312,1.214115,1.231996,449.000000,512.000000
75%,224931.500000,0.016177,0.000000,0.258513,1.252501,1.276550,627.000000,677.000000
max,299999.000000,0.151124,0.098301,0.823813,1.595928,1.623699,1872.000000,1650.000000


In [14]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.scatter(pred_df["y_true"], pred_df["y_pred"], alpha=0.2)
plt.xlabel("True Burn Probability")
plt.ylabel("Predicted Burn Probability")
plt.title("GNN Predictions vs True")
plt.show()

: 